In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split,cross_val_predict,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import recall_score,precision_score,accuracy_score,f1_score,confusion_matrix,roc_auc_score
import shap
import joblib

import warnings
warnings.filterwarnings('ignore')


In [2]:
# LOADING DATA

df = pd.read_csv(r'C:\Users\indur\OneDrive\Desktop\power bi projects\Public_Transport_Delay\ETL\-Smart-Traffic-Accident-Risk-Prediction-System-Using-Machine-Learning\DATA\feature_engineering.csv')
df


,Unnamed: 0,WEATHER_CONDITION,VISIBILITY,TRAFFIC_DENSITY,VEHICLE_SPEED,ROAD_CONDITION,HOUR,SPEED_LIMIT,ROAD_TYPE,PRECIPITATION,JUNCTION,TRAFFIC_SIGNAL,ACCIDENT_OCCURRENCE,RISK_SCORE,SPEED_RISK,LOW_VISIBILITY,IS_NIGHT,HIGH_TRAFFIC
0,0,Clearsky,4.5,Lowtraffic,31,Muddy,6,40,Serviceroad,Snowfall,0,1,0,0.55,1.29,0,0,0
1,1,Sunny,7.0,Meddensity,74,Icy,7,70,Expressway,Heavyrain,0,1,1,0.46,0.95,0,0,0
2,2,Storm,10.0,Light,110,Dryroad,12,50,Rural,Heavyrain,0,0,0,0.51,0.45,0,0,0
3,3,Clearsky,3.0,Meddensity,108,Waterwater,1,80,Expressway,Snowfall,1,0,0,0.48,0.74,0,1,0
4,4,Misty,13.8,Hgh,115,Muddy,7,60,Urbanroad,Snowfall,0,0,0,0.48,0.52,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,59995,Misty,10.0,High,10,Waterwater,12,50,Bridgelane,Fogdrizzle,0,0,0,0.42,5.00,0,0,1
59996,59996,Rainy,10.0,Medtraffic,100,Dryroad,23,50,Highway,Norain,1,0,0,0.46,0.50,0,1,1
59997,59997,Snow,9.1,Jampacked,59,Roughroad,12,70,City,Lightrain,1,0,1,0.50,1.19,0,0,1
59998,59998,Foggy,4.8,Low,49,Dryroad,3,100,Serviceroad,Lightrain,0,0,1,0.44,2.04,0,1,0


In [3]:
# FEATURES AND TARGET VALUES

x = df.drop('ACCIDENT_OCCURRENCE',axis=True)

y = df['ACCIDENT_OCCURRENCE']
df

,Unnamed: 0,WEATHER_CONDITION,VISIBILITY,TRAFFIC_DENSITY,VEHICLE_SPEED,ROAD_CONDITION,HOUR,SPEED_LIMIT,ROAD_TYPE,PRECIPITATION,JUNCTION,TRAFFIC_SIGNAL,ACCIDENT_OCCURRENCE,RISK_SCORE,SPEED_RISK,LOW_VISIBILITY,IS_NIGHT,HIGH_TRAFFIC
0,0,Clearsky,4.5,Lowtraffic,31,Muddy,6,40,Serviceroad,Snowfall,0,1,0,0.55,1.29,0,0,0
1,1,Sunny,7.0,Meddensity,74,Icy,7,70,Expressway,Heavyrain,0,1,1,0.46,0.95,0,0,0
2,2,Storm,10.0,Light,110,Dryroad,12,50,Rural,Heavyrain,0,0,0,0.51,0.45,0,0,0
3,3,Clearsky,3.0,Meddensity,108,Waterwater,1,80,Expressway,Snowfall,1,0,0,0.48,0.74,0,1,0
4,4,Misty,13.8,Hgh,115,Muddy,7,60,Urbanroad,Snowfall,0,0,0,0.48,0.52,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,59995,Misty,10.0,High,10,Waterwater,12,50,Bridgelane,Fogdrizzle,0,0,0,0.42,5.00,0,0,1
59996,59996,Rainy,10.0,Medtraffic,100,Dryroad,23,50,Highway,Norain,1,0,0,0.46,0.50,0,1,1
59997,59997,Snow,9.1,Jampacked,59,Roughroad,12,70,City,Lightrain,1,0,1,0.50,1.19,0,0,1
59998,59998,Foggy,4.8,Low,49,Dryroad,3,100,Serviceroad,Lightrain,0,0,1,0.44,2.04,0,1,0


In [4]:
df.columns

df.drop(columns=['Unnamed: 0'],axis=True)

,WEATHER_CONDITION,VISIBILITY,TRAFFIC_DENSITY,VEHICLE_SPEED,ROAD_CONDITION,HOUR,SPEED_LIMIT,ROAD_TYPE,PRECIPITATION,JUNCTION,TRAFFIC_SIGNAL,ACCIDENT_OCCURRENCE,RISK_SCORE,SPEED_RISK,LOW_VISIBILITY,IS_NIGHT,HIGH_TRAFFIC
0,Clearsky,4.5,Lowtraffic,31,Muddy,6,40,Serviceroad,Snowfall,0,1,0,0.55,1.29,0,0,0
1,Sunny,7.0,Meddensity,74,Icy,7,70,Expressway,Heavyrain,0,1,1,0.46,0.95,0,0,0
2,Storm,10.0,Light,110,Dryroad,12,50,Rural,Heavyrain,0,0,0,0.51,0.45,0,0,0
3,Clearsky,3.0,Meddensity,108,Waterwater,1,80,Expressway,Snowfall,1,0,0,0.48,0.74,0,1,0
4,Misty,13.8,Hgh,115,Muddy,7,60,Urbanroad,Snowfall,0,0,0,0.48,0.52,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,Misty,10.0,High,10,Waterwater,12,50,Bridgelane,Fogdrizzle,0,0,0,0.42,5.00,0,0,1
59996,Rainy,10.0,Medtraffic,100,Dryroad,23,50,Highway,Norain,1,0,0,0.46,0.50,0,1,1
59997,Snow,9.1,Jampacked,59,Roughroad,12,70,City,Lightrain,1,0,1,0.50,1.19,0,0,1
59998,Foggy,4.8,Low,49,Dryroad,3,100,Serviceroad,Lightrain,0,0,1,0.44,2.04,0,1,0


In [5]:
y = df['ACCIDENT_OCCURRENCE']
print(y.shape)
print(type(y))

(60000,)
<class 'pandas.core.series.Series'>


In [6]:
# num and categorical

num = x.select_dtypes(include=['int64']).columns
cat = x.select_dtypes(include='object').columns

In [7]:
#pipe line for numerical columns

num_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])



In [8]:
# pipe line fro categorical columns 

cat_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('one_hot',OneHotEncoder(handle_unknown='ignore'))
])

In [9]:
#   preprocessing using COLUMN TRANSFORM

preprocessing = ColumnTransformer([

    ('num_col',num_pipeline,num),
    ('cat_col',cat_pipeline,cat)
])



In [10]:
x = preprocessing.fit_transform(x)

In [11]:
# splitting the data 
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [12]:
#  MODEL PIPELINES LOGISTIC REGRESSION

models = Pipeline([
    ('logistic',LogisticRegression(random_state=42)),
    
])

random_model = Pipeline([
    ('RFM',RandomForestClassifier())
])

xg_model = Pipeline([
    ('XGB',XGBClassifier())
])


In [ ]:
# random forest cross_valuation

RB_cross_valudation = cross_val_predict(random_model,x,y,cv=5)
print(RB_cross_valudation)


In [ ]:
# XGBOOST cross_valudataion

XGB_cross_valudatation = cross_val_predict(xg_model,x,y,cv=5)
print(XGB_cross_valudatation)